In [1]:
%load_ext autoreload
%autoreload 2
import os
import re 
import sys
import numpy as np
import pandas as pd
import xarray as xr
from os.path import join as pjoin
from tqdm.notebook import tqdm
from sklearn.metrics import mutual_info_score
import plotly.graph_objects as go
from scipy.stats import pearsonr, spearmanr, zscore
from natsort import natsorted
import statsmodels.api as sm
from statsmodels.formula.api import ols
from numpy.random import RandomState, SeedSequence, MT19937

sys.path.append('../')
import circletrack_behavior as ctb
import circletrack_neural as ctn
import place_cells as pc
import plotting_functions as pf

In [2]:
## Settings
project_folder = ['MultiCon_Imaging']
experiment_folders = ['MultiCon_Imaging5', 'MultiCon_Imaging6', 'MultiCon_Imaging7']
dpath = f'../../{project_folder[0]}'
fig_path = f'../../../Manuscripts/MultiCon/intermediate_plots/first_days_place_cells'
int_data = f'../../../Manuscripts/MultiCon/intermediate_plots/intermediate_data'
chance_color = '#7d7d7d'
avg_color = '#287347'
subject_color = '#7d7d7d'
ce_colors = ['#7A22BC', '#378616']
ce_colors_dict = {'Two-context': '#378616', 'Multi-context': '#7A22BC'}
symbol_dict = {'Two-context': 'x', 'Multi-context': 'circle'}
symbols_list = ['x', 'circle']
context_colors = {'A': '#00802d', 'B': '#006c79', 'C': '#004da4', 'D': '#430073'}
mouse_colors = ['midnightblue', 'darkred', 'darkorchid', 'darkturquoise']
session_list = [f'A{x}' for x in np.arange(1, 6)] + [f'B{x}' for x in np.arange(1, 6)] + [f'C{x}' for x in np.arange(1, 6)] + [f'D{x}' for x in np.arange(1, 6)]
control_list = [f'A{x}' for x in np.arange(1, 16)] + [f'B{x}' for x in np.arange(1, 6)]
day_list = [f'Day {x}' for x in np.arange(1, 21)]
bin_size = 0.16 ## size of linear position bins equivalent to 2cm-wide bins
velocity_thresh = 10
centroid_distance = 4
data_of_interest = 'aligned_place_cells' ## one of behav, aligned_minian, aligned_place_cells, lin_behav
data_type = 'S'
crossreg_str = None

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

xr.set_options(keep_attrs=True)

rs = RandomState(MT19937(SeedSequence(24601)))

### Example mouse. Cross-register between day 15 and 16 and see if those cells increase/decrease in spatial information.

In [ ]:
## Set mouse information
experiment = 'MultiCon_Imaging6'
mouse = 'mc58'
days_of_int = ['15', '16']
crossreg_path = f'../../../CircleTrack/{project_folder[0]}/{experiment}/output/cross_registration_results/circletrack_data/{mouse}'

cell_dict = {'mouse': [], 'group': [], 'sex': [], '0_percent': [], '1_percent': [], '2_percent': [], '3_percent': [], '4_percent': []}
## Select cells that are cross-registered across all four days
mappings = pd.read_pickle(pjoin(crossreg_path, f'mappings_{centroid_distance}_{crossreg_str}.pkl'))
date_list = []
for idx, d in enumerate(days_of_int):
    session = f'{mouse}_{data_type}_{d}.nc'
    exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/{mouse}/{data_type}')

    S = xr.open_dataset(pjoin(exp_path, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced and thresholded activity
    date_list.append(S.attrs['date'])

shared_cells = mappings['session'][date_list].dropna().reset_index(drop=True)
si_ar = np.zeros((shared_cells.shape[0], len(days_of_int)))
place_ar = np.zeros((shared_cells.shape[0], len(days_of_int)))
for idx, d in enumerate(days_of_int):
    session = f'{mouse}_{data_type}_{d}.nc'
    exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/{mouse}/{data_type}')

    S = xr.open_dataset(pjoin(exp_path, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced and thresholded activity
    sdata = S.sel(unit_id=shared_cells[S.attrs['date']].to_numpy())
    si_ar[:, idx] = sdata['skaggs_info'].values
    place_ar[:, idx] = sdata['skaggs_place'].values

# place_cells = np.sum(place_ar, axis=1)
# values, counts = np.unique(place_cells, return_counts=True)
# norm_counts = (counts / np.sum(counts)) * 100

# cell_dict['mouse'].append(mouse)
# cell_dict['group'].append(S.attrs['group'])
# cell_dict['sex'].append(S.attrs['sex'])
# for idx in np.arange(0, 5):
#     cell_dict[f'{idx}_percent'].append(norm_counts[idx])

In [11]:
shared_cells

,2025_02_01,2025_02_02
0,40.0,54.0
1,179.0,226.0
2,314.0,368.0
3,296.0,356.0
4,168.0,206.0
...,...,...
179,337.0,381.0
180,340.0,386.0
181,114.0,144.0
182,70.0,87.0
